In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv('gold_churn_data.csv')
df.head()

,Unnamed: 0,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_years,spend_per_month
0,0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0.083333,29.850000
1,1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,...,No,No,One year,No,Mailed check,56.95,1889.50,No,2.833333,55.573529
2,2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0.166667,54.075000
3,3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,3.750000,40.905556
4,4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0.166667,75.825000


In [4]:
df = df.iloc[:,1:]

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder


# Creating X and y
X = df.drop("Churn", axis=1)
y = df["Churn"]
# Step 1: Drop the 'customerID' column
X = X.drop(columns=['customerID'])

# Step 2: Convert 'TotalCharges' to numeric (handles spaces or non-numeric values)
X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors='coerce')

# Step 3: Convert target column 'y' to binary values
y = y.map({'Yes': 1, 'No': 0})

# Step 4: Identify column types
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Step 5: Define preprocessing pipeline (no model yet)
preprocessor = ColumnTransformer(transformers=[
('num', SimpleImputer(strategy='mean'), numerical_cols),
('cat', OneHotEncoder(handle_unknown='ignore', sparse_output =False),categorical_cols)
])


In [6]:
X_cleaned = preprocessor.fit_transform(X)

In [7]:
X_cleaned

array([[  0.  ,   1.  ,  29.85, ...,   0.  ,   1.  ,   0.  ],
       [  0.  ,  34.  ,  56.95, ...,   0.  ,   0.  ,   1.  ],
       [  0.  ,   2.  ,  53.85, ...,   0.  ,   0.  ,   1.  ],
       ...,
       [  0.  ,  11.  ,  29.6 , ...,   0.  ,   1.  ,   0.  ],
       [  1.  ,   4.  ,  74.4 , ...,   0.  ,   0.  ,   1.  ],
       [  0.  ,  66.  , 105.65, ...,   0.  ,   0.  ,   0.  ]])

In [8]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=500,
    max_depth=8,          # try: 5, 10, 15, None
    min_samples_split=5,     # try: 2, 5, 10
    min_samples_leaf=2,      # try: 1, 2, 4
    max_features="sqrt",     # try: "sqrt", "log2", 0.5
    class_weight="balanced", # useful if churners are fewer
    random_state=42,
    n_jobs=-1
)

In [9]:
model.fit(X_cleaned, y)

RandomForestClassifier(class_weight='balanced', max_depth=8, min_samples_leaf=2,
                       min_samples_split=5, n_estimators=500, n_jobs=-1,
                       random_state=42)

In [10]:
model.feature_importances_

array([0.00511127, 0.09166339, 0.05097615, 0.06969664, 0.10057161,
       0.0477349 , 0.00418546, 0.00415963, 0.00431361, 0.00459861,
       0.00419033, 0.00481149, 0.00213139, 0.00194003, 0.00393033,
       0.00183837, 0.00431763, 0.01901654, 0.04258932, 0.01040701,
       0.05918093, 0.00767491, 0.01106746, 0.01699092, 0.00965096,
       0.00577768, 0.00986875, 0.00838919, 0.00356511, 0.04994696,
       0.00682708, 0.01229877, 0.00401633, 0.01073044, 0.00453336,
       0.00438819, 0.00932291, 0.00518205, 0.13875341, 0.02029188,
       0.06339915, 0.00783298, 0.0075583 , 0.00404754, 0.00452386,
       0.0319966 , 0.00400054])

In [14]:
import joblib

joblib.dump(preprocessor, '../app/transformer.pkl')

['../app/transformer.pkl']

In [13]:
joblib.dump(model, '../app/model.pkl')

['../app/model.pkl']